# Recycling Robot — Chapter 3 MDP

Strictly focused on the Chapter 3 MDP concepts: states, actions, transition probabilities, rewards, stochastic policy, and the Bellman expectation equations for $V_\pi$ and $Q_\pi$.

No Monte Carlo returns, policy iteration, or softmax policy improvement.

In [1]:
# States
states = ["high", "low"]

# Available actions in each state
actions = {
    "high": ["search", "wait"],
    "low": ["search", "wait", "recharge"],
}


In [2]:
# Transition probabilities: P(s' | s, a)
alpha = 0.8
beta = 0.7

P = {
    ("high", "search"): {"high": alpha, "low": 1 - alpha},
    ("high", "wait"): {"high": 1.0},
    ("low", "search"): {"low": beta, "high": 1 - beta},
    ("low", "wait"): {"low": 1.0},
    ("low", "recharge"): {"high": 1.0},
}


In [3]:
# Reward function: R(s, a, s')
r_search = 1.0
r_wait = 0.2

R = {
    ("high", "search", "high"): r_search,
    ("high", "search", "low"): r_search,
    ("high", "wait", "high"): r_wait,
    ("low", "search", "low"): r_search,
    ("low", "search", "high"): -3.0,
    ("low", "wait", "low"): r_wait,
    ("low", "recharge", "high"): 0.0,
}


## Stochastic Policy $\pi(a|s)$

The policy gives a probability for each action available in a state. The probabilities for each state must sum to 1.

In [4]:
stochastic_policy = {
    "high": {
        "search": 1 / 2,
        "wait": 1 / 2,
    },
    "low": {
        "search": 1 / 3,
        "wait": 1 / 3,
        "recharge": 1 / 3,
    },
}

# Verify that each state's action probabilities sum to 1.
for state in states:
    probability_sum = sum(stochastic_policy[state].values())
    assert abs(probability_sum - 1.0) < 1e-9

stochastic_policy


{'high': {'search': 0.5, 'wait': 0.5},
 'low': {'search': 0.3333333333333333,
  'wait': 0.3333333333333333,
  'recharge': 0.3333333333333333}}

## Discount factor

$\gamma$ determines how much future rewards contribute to the current value.

In [5]:
gamma = 0.9


## $Q_\pi(s,a)$ — Action-value function

For the current policy $\pi$:

$$
q_\pi(s,a) = \sum_{s'} p(s',r|s,a)\left[r + \gamma v_\pi(s')\right]
$$

Here the transition model and reward model are known, so the expectation can be calculated directly.

In [12]:
def calculate_Q_pi(state, action, V_pi):
    """Calculate q_pi(s, a) using the known MDP model."""
    value = 0.0

    for next_state, probability in P[(state, action)].items():
        reward = R[(state, action, next_state)]
        value += probability * (reward + gamma * V_pi[next_state])

    return value


## $V_\pi(s)$ — State-value function

The Bellman expectation equation is:

$$
v_\pi(s) = \sum_a \pi(a|s)q_\pi(s,a)
$$

Equivalently, this is the expected return from state $s$ when following the stochastic policy $\pi$.

In [13]:
def calculate_V_pi(V_pi):
    """One Bellman expectation update for all states."""
    updated_V = {}

    for state in states:
        state_value = 0.0

        for action, action_probability in stochastic_policy[state].items():
            q_value = calculate_Q_pi(state, action, V_pi)
            state_value += action_probability * q_value

        updated_V[state] = state_value

    return updated_V


## Iterative evaluation of the fixed stochastic policy

Starting from an initial value estimate, repeatedly apply the Bellman expectation equation until the values stabilize.

The policy itself is **not changed** here.

In [14]:
V_pi = {state: 0.0 for state in states}

theta = 1e-6
max_iterations = 1000

for iteration in range(max_iterations):
    updated_V = calculate_V_pi(V_pi)

    delta = max(
        abs(updated_V[state] - V_pi[state])
        for state in states
    )

    V_pi = updated_V

    if delta < theta:
        break

print(f"Converged after {iteration + 1} iterations")
print("V_pi:")
for state in states:
    print(f"  V_pi({state}) = {V_pi[state]:.6f}")


Converged after 126 iterations
V_pi:
  V_pi(high) = 5.068957
  V_pi(low) = 4.034474


## Calculate $Q_\pi(s,a)$ for every state-action pair

In [15]:
Q_pi = {}

for state in states:
    Q_pi[state] = {}

    for action in actions[state]:
        Q_pi[state][action] = calculate_Q_pi(
            state=state,
            action=action,
            V_pi=V_pi,
        )

print("Q_pi:")
for state in states:
    print(f"\nState: {state}")
    for action, value in Q_pi[state].items():
        print(f"  Q_pi({state}, {action}) = {value:.6f}")


Q_pi:

State: high
  Q_pi(high, search) = 5.375855
  Q_pi(high, wait) = 4.762061

State: low
  Q_pi(low, search) = 3.710337
  Q_pi(low, wait) = 3.831027
  Q_pi(low, recharge) = 4.562061


## Verify the Bellman expectation equation

For each state, the state value should equal the policy-weighted average of its action values:

$$
V_\pi(s)=\sum_a \pi(a|s)Q_\pi(s,a)
$$


In [16]:
for state in states:
    reconstructed_value = sum(
        stochastic_policy[state][action] * Q_pi[state][action]
        for action in actions[state]
    )

    print(f"State: {state}")
    print(f"  V_pi(s)                 = {V_pi[state]:.6f}")
    print(f"  Σ π(a|s) Q_pi(s,a)     = {reconstructed_value:.6f}")
    print()


State: high
  V_pi(s)                 = 5.068957
  Σ π(a|s) Q_pi(s,a)     = 5.068958

State: low
  V_pi(s)                 = 4.034474
  Σ π(a|s) Q_pi(s,a)     = 4.034475



## Optional: sample one action from the stochastic policy

This only demonstrates how $\pi(a|s)$ can be used to select an action. It does not update the policy.

In [20]:
import random

def choose_action(state):
    available_actions = list(stochastic_policy[state].keys())
    probabilities = list(stochastic_policy[state].values())

    return random.choices(
        available_actions,
        weights=probabilities,
        k=1,
    )[0]


for state in states:
    print(f"State: {state} -> sampled action: {choose_action(state)}")

stochastic_policy

State: high -> sampled action: wait
State: low -> sampled action: recharge


{'high': {'search': 0.5, 'wait': 0.5},
 'low': {'search': 0.3333333333333333,
  'wait': 0.3333333333333333,
  'recharge': 0.3333333333333333}}

In [23]:
# Optimal value function

def calculate_V_star(V_star):
    """
    Bellman optimality equation:

    V*(s) = max_a Q*(s, a)
    """
    updated_V_star = {}

    for state in states:
        action_values = []

        for action in actions[state]:
            q_value = 0.0

            for next_state, probability in P[(state, action)].items():
                reward = R[(state, action, next_state)]

                q_value += probability * (
                    reward + gamma * V_star[next_state]
                )

            action_values.append(q_value)

        updated_V_star[state] = max(action_values)

    return updated_V_star


# Then iteratively apply it:
V_star = {state: 0.0 for state in states}

theta = 1e-6
max_iterations = 1000

for iteration in range(max_iterations):

    updated_V_star = calculate_V_star(V_star)

    delta = max(
        abs(updated_V_star[state] - V_star[state])
        for state in states
    )

    V_star = updated_V_star

    if delta < theta:
        break

print(f"Converged after {iteration + 1} iterations")

for state in states:
    print(f"V*({state}) = {V_star[state]:.6f}")
    

# Now calculate the optimal \(Q\):
Q_star = {}

for state in states:
    Q_star[state] = {}

    for action in actions[state]:

        q_value = 0.0

        for next_state, probability in P[(state, action)].items():
            reward = R[(state, action, next_state)]

            q_value += probability * (
                reward + gamma * V_star[next_state]
            )

        Q_star[state][action] = q_value


for state in states:
    print(f"\nState: {state}")

    for action, value in Q_star[state].items():
        print(f"Q*({state}, {action}) = {value:.6f}")

# Finally, extract the optimal policy:

optimal_policy = {}

for state in states:

    best_action = max(
        Q_star[state],
        key=Q_star[state].get
    )

    optimal_policy[state] = best_action


print("Optimal Policy:")

for state, action in optimal_policy.items():
    print(f"π*({state}) = {action}")


Converged after 131 iterations
V*(high) = 8.474568
V*(low) = 7.627110

State: high
Q*(high, search) = 8.474569
Q*(high, wait) = 7.827111

State: low
Q*(low, search) = 6.893213
Q*(low, wait) = 7.064399
Q*(low, recharge) = 7.627111
Optimal Policy:
π*(high) = search
π*(low) = recharge
